In [ ]:
!git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
# !git checkout 329f99d3ad99ecb2b13be93958b845d4326fe153
!chmod +x studio/setup.sh && ./studio/setup.sh

Cloning into 'unsloth'...
remote: Enumerating objects: 1307, done.
remote: Counting objects: 100% (1307/1307), done.
remote: Compressing objects: 100% (1124/1124), done.
remote: Total 1307 (delta 186), reused 801 (delta 160), pack-reused 0 (from 0)
Receiving objects: 100% (1307/1307), 33.51 MiB | 14.13 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/unsloth

  🦥 Unsloth Studio Setup
  ────────────────────────────────────────────────────
                 upgrading npm...
  node           v20.19.0 | npm 11.14.1
                 installing bun...
                 bun installed (1.3.13)
                 building frontend...
                 using bun for package install (faster)
  frontend       built
                 Colab detected, installing Studio backend dependencies...
  python         backend deps installed into system Python
                 continuing to llama.cpp install for GGUF inference support
  deps           [====================] 14/14  ROCm torch (final)  
 

In [ ]:
import sys, time
sys.path.insert(0, "/content/unsloth/studio/backend")
from colab import start
start()

2026-05-10 11:07:10 [info     ] 🦥 Starting Unsloth Studio...
2026-05-10 11:07:10 [info     ]    Loading backend...
2026-05-10 11:07:10 [info     ]    Starting server...


INFO:     Started server process [2601]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8888 (Press CTRL+C to quit)


Hardware detected: CUDA -- NVIDIA L4

DEFAULT ADMIN ACCOUNT CREATED
    username: unsloth
    password saved to: /root/.unsloth/studio/auth/.bootstrap_password
    Open the Studio UI to sign in and change it.

{"timestamp": "2026-05-10T11:07:16.540824Z", "level": "info", "event": "Pre-caching helper GGUF: unsloth/gemma-4-E2B-it-GGUF/gemma-4-E2B-it-UD-Q4_K_XL.gguf"}
{"timestamp": "2026-05-10T11:07:18.499247Z", "level": "info", "event": "   Server started!"}


# Fine-Tune Config — focal_finetune_train.jsonl

## Dataset
- 1,131 training records
- Max seq length observed: ~1,500 tokens (P90: ~1,179)
- Tool-calling task, 5 tools, 2:1 noise:matters class balance

## Hyperparameters

### LoRA
| Setting | Value |
|---|---|
| `lora_rank` | 16 |
| `lora_alpha` | 32 |
| `lora_dropout` | 0.05 |
| `target_modules` | all-linear |
| `use_rslora` | false |

### Training
| Setting | Value |
|---|---|
| `epochs` | 2 |
| `batch_size` | 2 |
| `gradient_accumulation_steps` | 4 |
| **Effective batch size** | **8** |
| `learning_rate` | 2e-4 (0.0002) |
| `optimizer` | AdamW 8-bit |
| `lr_scheduler_type` | cosine |
| `warmup_steps` | 15 |
| `weight_decay` | 0.01 |
| `max_seq_length` | 2048 |
| `bf16` | true (fp16 if on T4) |

### Eval / Save (for best-checkpoint loading)
| Setting | Value |
|---|---|
| `eval_strategy` | steps |
| `eval_steps` | 0.1 (resolves to 28) |
| `save_strategy` | steps |
| `save_steps` | 28 |
| `load_best_model_at_end` | true |
| `metric_for_best_model` | eval_loss |

## Step Math

```
Samples:                 1,131
Effective batch:         2 × 4 = 8
Steps per epoch:         1,131 ÷ 8 ≈ 142
Total steps (2 epochs):  284

Warmup steps:            15        (≈ 5% of 284)
Eval / Save every:       28 steps  (10 evaluations during run)
```

- **2 epochs**: enough to learn the schema, not enough to memorize 1,131 samples
- **Rank 16**: small adapter capacity, can't memorize the dataset
- **Dropout 0.05 + weight_decay 0.01**: dual regularization

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8888, height = 1200, width = "100%")
for _ in range(10000): time.sleep(300), print("=", end = "")

<IPython.core.display.Javascript object>

{"timestamp": "2026-05-10T11:07:27.281795Z", "level": "info", "event": "Helper GGUF cached: 1 file(s)"}
{"timestamp": "2026-05-10T11:07:27.673780Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/", "status_code": 200, "process_time_ms": 1.71}
{"timestamp": "2026-05-10T11:07:28.408789Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/api/health", "status_code": 200, "process_time_ms": 0.91}
{"timestamp": "2026-05-10T11:07:28.522447Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/api/auth/status", "status_code": 200, "process_time_ms": 2.19}
{"timestamp": "2026-05-10T11:07:28.631538Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/api/auth/status", "status_code": 200, "process_time_ms": 1.7}
{"timestamp": "2026-05-10T11:07:29.054690Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/api/auth/status", "status_code": 200, "process_time_ms": 1.73}
{"times

Generating train split: 0 examples [00:00, ? examples/s]

{"timestamp": "2026-05-10T11:08:53.495927Z", "level": "info", "event": "Format check result: requires_mapping=True, format=unknown, is_image=False"}
{"timestamp": "2026-05-10T11:08:53.506805Z", "level": "info", "event": "request_completed", "method": "POST", "path": "/api/datasets/check-format", "status_code": 200, "process_time_ms": 958.02}
{"timestamp": "2026-05-10T11:08:58.028988Z", "level": "info", "event": "request_completed", "method": "GET", "path": "/api/datasets/local", "status_code": 200, "process_time_ms": 2.13}
{"timestamp": "2026-05-10T11:08:58.270957Z", "level": "info", "event": "request_completed", "method": "POST", "path": "/api/datasets/upload", "status_code": 200, "process_time_ms": 194.49}
{"timestamp": "2026-05-10T11:09:00.574650Z", "level": "info", "event": "Checking format for dataset: /root/.unsloth/studio/assets/datasets/uploads/775d33f250a846e896e5651773ed1391_focal_finetune_train.jsonl"}
{"timestamp": "2026-05-10T11:09:01.281625Z", "level": "info", "event": "F

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
from pathlib import Path

src = Path("/root/.unsloth/studio/exports/unsloth_gemma-4-E2B-it_1778411897/checkpoint-284")
dst = Path("/content/drive/MyDrive/unsloth_exports/gemma-4-E2B-it")

dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved to:", dst)

Saved to: /content/drive/MyDrive/unsloth_exports/gemma-4-E2B-it


In [ ]:
import shutil
from pathlib import Path

drive_model = Path("/root/.unsloth/studio/exports/gemma-4-E2B-it-gguf")
local_model = Path("/content/drive/MyDrive/unsloth_exports/gemma-4-E2B-it-gguf-1")

if local_model.exists():
    shutil.rmtree(local_model)

shutil.copytree(drive_model, local_model)

print("Local model copied to:", local_model)

Local model copied to: /content/drive/MyDrive/unsloth_exports/gemma-4-E2B-it-gguf-1


In [ ]:
!ls -lrth /root/.unsloth/studio/exports/gemma-4-E2B-it-gguf

total 9.6G
-rw-r--r-- 1 root root 942M May 10 11:52 gemma-4-e2b-it.BF16-mmproj.gguf
-rw-r--r-- 1 root root 8.7G May 10 11:53 gemma-4-e2b-it.F16.gguf
-rw-r--r-- 1 root root   44 May 10 11:53 export_metadata.json


## Litert torch doesnt have proper support to convert gemma4 e2b model to litertlm the below code is NO op and was used to experiement if it would work with litert-lm 0.11.0 on android

In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 183.9 MB/s eta 0:00:00


In [ ]:
!uv tool install litert-torch-nightly

Resolved 83 packages in 399ms
Prepared 83 packages in 16.80s
Installed 83 packages in 238ms
 + absl-py==2.4.0
 + ai-edge-litert-nightly==2.2.0.dev20260506
 + ai-edge-quantizer-nightly==0.7.0.dev20260507
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + backports-strenum==1.3.1
 + certifi==2026.4.22
 + charset-normalizer==3.4.7
 + click==8.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + filelock==3.29.0
 + fire==0.7.1
 + flatbuffers==25.12.19
 + fsspec==2026.4.0
 + h11==0.16.0
 + hf-xet==1.5.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.14.0
 + idna==3.13
 + immutabledict==4.2.1
 + jax==0.10.0
 + jaxlib==0.10.0
 + jaxtyping==0.3.9
 + jinja2==3.1.6
 + kagglehub==1.0.1
 + kagglesdk==0.1.23
 + lark==1.3.1
 + litert-converter==0.1.0
 + litert-torch-nightly==0.10.0.dev20260506
 + markdown-it-py==4.1.0
 + markupsafe==3.0.3
 + mdurl==0.1.2
 + ml-dtypes==0.5.4
 + mpmath==1.3.0
 + multipledispatch==1.0.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cubla

In [ ]:
!/root/.local/bin/litert-torch export_hf \
  --model=/content/gemma-4-E2B-it \
  --output_dir=/content/gemma4_e2b \
  --externalize_embedder \
  --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm \
  --quantization_recipe dynamic_wi4_afp32 \
  --cache_length 32768

I0000 00:00:1778145191.393276    3255 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/root/.local/share/uv/tools/litert-torch-nightly/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/root/.local/share/uv/tools/litert-torch-nightly/lib/python3.12/site-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.
============== Export Configur

In [ ]:
!ls -lrth /content/gemma4_e2b/model.litertlm

-rw-r--r-- 1 root root 2.4G May  7 09:17 /content/gemma4_e2b/model.litertlm


In [ ]:
!md5sum /content/drive/MyDrive/unsloth_exports/gemma-4-E2B-int4-it.litertlm

c173f6dc3392d0ebabf726fdbd9f353b  /content/drive/MyDrive/unsloth_exports/gemma-4-E2B-int4-it.litertlm


In [ ]:
!ls -lh /content/litertlm_export_e2b_int4_4k

total 24G
-rw-r--r-- 1 root root  12K May  1 16:49 chat_template.jinja
-rwxr-xr-x 1 root root 199M May  1 16:43 embedder_quantized.tflite
-rw-r--r-- 1 root root 1.6G May  1 16:43 embedder.tflite
-rw-r--r-- 1 root root  12K May  1 16:49 llm_metadata.pb
-rw-r--r-- 1 root root 2.5G May  1 16:49 model.litertlm
-rwxr-xr-x 1 root root 1.2G May  1 16:43 model_quantized.tflite
-rw-r--r-- 1 root root 8.5G May  1 16:42 model.tflite
-rwxr-xr-x 1 root root 1.1G May  1 16:49 per_layer_embedder_quantized.tflite
-rw-r--r-- 1 root root 8.8G May  1 16:45 per_layer_embedder.tflite
-rw-r--r-- 1 root root 2.7K May  1 16:49 tokenizer_config.json
-rw-r--r-- 1 root root  31M May  1 16:49 tokenizer.json


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/gemma4_e2b/model.litertlm")
dst = Path("/content/drive/MyDrive/unsloth_exports/gemma-4-E2B-int4-def-it.litertlm")

dst.parent.mkdir(parents=True, exist_ok=True)

assert src.exists(), "model.litertlm not found"
shutil.copy2(src, dst)

print("Copied to:", dst)

Copied to: /content/drive/MyDrive/unsloth_exports/gemma-4-E2B-int4-def-it.litertlm


In [ ]:
!/root/.local/bin/litert-torch export_hf \
  --model=/root/.unsloth/studio/exports/unsloth_gemma-4-E2B-it_1778088714/checkpoint-243 \
  --output_dir=/content/gemma4_e2b \
  --externalize_embedder \
  --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm \
  --quantization_recipe dynamic_wi4_afp32 \
  --cache_length 8192

In [ ]:
from google.colab import drive
drive.unmount()

AttributeError: module 'google.colab.drive' has no attribute 'unmount'

In [ ]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy
